# 00 · Collect with Apify (manual pilot)

Run this notebook in your **CPU workspace on the seminar server**. The server sends API requests; Actors execute in Apify's cloud. No GPU or extra Python packages are needed.

Choose a keyword search, hashtag collection, or comments on full post URLs. Each collection stores the exact input, run/build IDs, status, original dataset rows and a checksum in your private home folder. Results stay outside the public repository and the shared corpus.

This notebook collects metadata or comments and can prepare videos, covers and slideshow images for the existing corpus submission form. Private exports stay intact; staging into incoming is a separate action. Scheduling and a dedicated collection form remain future work.


## 1. Configure one collection

Use a new `COLLECTION_NAME` for an intentional new run. Reusing the directory is refused to avoid accidental repeat charges. `LIMIT` is per query/hashtag or per post for comments, not a global count. `REPLIES = 0` skips replies. Requested counts are limits, not completeness guarantees; a successful run can still contain errors or fewer results.

Set your own `MAX_COST_USD` after checking the Actor's current pricing. The request passes this ceiling and a run timeout to Apify. Labels may eventually be public: do not use student names.


In [ ]:
from pathlib import Path
from getpass import getpass
from apify_collection import make_input, start_run, fetch_run
from apify_prepare import prepare_run, stage_run

MODE = "search"  # "search", "hashtag", or "comments"
VALUES = []  # e.g. ["climate change"], ["climatechange"], or full TikTok post URLs
LABEL = "Climate"
COLLECTION_NAME = "climate-pilot-01"
LIMIT = 20
REPLIES = 0
DOWNLOAD_MEDIA = True  # Apify media add-ons may be charged; included in your run budget
MAX_COST_USD = None  # Enter your chosen maximum run charge in USD
TIMEOUT_SECONDS = 600
START_NEW_RUN = False  # Set True only when you intend to start a paid run

if not COLLECTION_NAME or Path(COLLECTION_NAME).name != COLLECTION_NAME or COLLECTION_NAME in {".", ".."}:
    raise ValueError("Use a single folder name for COLLECTION_NAME")
DIRECTORY = Path.home() / "apify-collections" / COLLECTION_NAME


## 2. Review the request and enter your token

Use your own Apify account token in the hidden prompt. Never paste it into a code cell, URL, output, chat, or a shared `.env` file. It stays in kernel memory. Restart the kernel when finished.


In [ ]:
run_input = make_input(MODE, VALUES, limit=LIMIT, replies=REPLIES, download_media=DOWNLOAD_MEDIA)
print({"mode": MODE, "label": LABEL, "input": run_input, "max_cost_usd": MAX_COST_USD})
token = getpass("Your Apify API token: ")


## 3. Start once

This is the chargeable step. It saves the request before making the API call and saves the returned run ID immediately. Run the next cell again to check progress. Closing or interrupting the notebook **does not stop the cloud run**; abort it in Apify Console if needed.

If starting returns a connection error, **do not remove the folder and retry**. Check Apify Console for the run; use its ID in the recovery field below. There is no automatic retry of a start request.


In [ ]:
if START_NEW_RUN:
    START_NEW_RUN = False  # Disarm this cell before making the network request
    receipt = start_run(
        token, DIRECTORY, mode=MODE, values=VALUES, label=LABEL,
        limit=LIMIT, replies=REPLIES, max_cost_usd=MAX_COST_USD,
        timeout_seconds=TIMEOUT_SECONDS, download_media=DOWNLOAD_MEDIA,
    )
    print({"run_id": receipt["id"], "status": receipt["status"]})
else:
    print("No new run started. Use the next cell to check or download an existing run.")


## 4. Check status / download results

Rerun while the run is active. Once it finishes, this cell downloads all pages into `items.jsonl`, retaining error rows and replies exactly as returned. Failed, aborted and timed-out runs can still be exported, explicitly marked partial. Inspect `error_items` even when status is `SUCCEEDED`.

After a kernel restart, rerun configuration and token entry with `START_NEW_RUN = False`, then this cell. An interrupted download restarts from the dataset's first page without starting another Actor. Existing completed exports are checksum-verified and reused. These are snapshots. The preparation step reuses existing corpus posts and preserves new observations separately.


In [ ]:
RECOVER_RUN_ID = None  # Only needed if the start response/receipt was lost
if DIRECTORY.exists():
    summary = fetch_run(token, DIRECTORY, run_id=RECOVER_RUN_ID)
    print(summary)
    if summary["exported"]:
        print(f"Saved privately to: {DIRECTORY}")
else:
    print("No collection directory yet. Configure and start a run first.")


## Files and next steps

- `request.json`: collection label, mode, exact Actor input, requested limits and time.
- `run.json`: run/build identifiers, status, dataset/storage IDs and reported usage.
- `items.jsonl`: original dataset records (TikTok IDs must remain strings in downstream tables).
- `export.json`: export timestamp, item/error counts, run status and SHA-256 checksum.

Keep all four together. Review failed items and coverage before analysis. Comments are separate observations linked to posts; they should later become a dedicated `comments.csv`, preserving reply relationships where available.

Use the preparation and staging cells below to create a submission. Raw Apify exports remain private; corpus attachments retain source fields with access URLs and credential-like keys removed. Comment runs require parent posts already in raw. Replies retain their parent IDs when supplied. The table builder produces a separate comments.csv. Clear notebook outputs before committing or sharing the notebook.

References: [Actor inputs](https://apify.com/clockworks/tiktok-scraper/input-schema), [comment inputs](https://apify.com/clockworks/tiktok-comments-scraper/input-schema), [start-run API](https://docs.apify.com/api/v2/actors-runs-post), [dataset export API](https://docs.apify.com/api/v2/dataset-items-get).


## 5. Prepare an ingestion batch privately

Run this after export has finished. Requires `ffmpeg` and `ffprobe` (already installed on the seminar server). It downloads videos, converts covers/slides to JPEG, and maps metadata to the corpus format. Existing raw metadata is reused byte-for-byte; newer observations are kept in the source attachment. Existing media is reused without downloading. Set `INCLUDE_MEDIA = False` deliberately for metadata-only imports. Expired URLs may require a new collection run.

Preparation retries reuse completed private media downloads. A failed post is excluded as a whole; review the error count before staging. Partial Actor runs require explicit opt-in. Unsuccessful preparations are never publishable. A preparation report stays private.


In [ ]:
INCLUDE_MEDIA = True
ALLOW_PARTIAL_RUN = False
RAW = Path("/corpus/raw")

preparation = prepare_run(
    DIRECTORY, raw=RAW, token=token, include_media=INCLUDE_MEDIA,
    allow_partial=ALLOW_PARTIAL_RUN,
)
print({"posts": len(preparation["posts"]), "comments": preparation["comments"],
       "skipped_records": len(preparation["errors"]), "partial_run": preparation["partial_run"]})


## 6. Stage for the existing form

Enter your seminar account folder name, then explicitly enable staging after reviewing preparation. This copies completed files into `/corpus/incoming/<account>/apify-<run-id>/`. Incoming is readable by other students under the existing server policy. Do not include student names in collection labels or queries. Private source files are retained.

Staging does **not** import anything. Wait at least two minutes after staging, then open `/services/corpus/`, select the batch, choose its label and collection strategy, preview and submit. For hashtag runs choose `hashtag` (requires the server form update). For comments, select the sampling strategy of the parent posts; `comments` describes the data type, not sampling. JSONL attachments are archived in `raw/_collections/`. Rebuild `04_build_tables` afterwards for comments.csv.

A run can be staged only once. After an interrupted staging operation inspect `staged.json` and your incoming folder; do not blindly delete receipts and repeat a submission.


In [ ]:
INCOMING_ACCOUNT = ""  # Your existing seminar account folder
STAGE_FOR_IMPORT = False

if STAGE_FOR_IMPORT:
    STAGE_FOR_IMPORT = False
    import re
    if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9._-]{0,63}", INCOMING_ACCOUNT):
        raise ValueError("Enter your seminar account folder name")
    batch_folder = stage_run(DIRECTORY, Path("/corpus/incoming") / INCOMING_ACCOUNT)
    print(f"Ready for the corpus form: {batch_folder.name}. Wait two minutes, then preview and submit.")
else:
    print("Nothing staged. Review preparation before enabling this step.")
